# Neural Operator Benchmark on Non-Euclidean Domains

Same protocol as `benchmark.ipynb`, but the three PDEs now live on two curved,
closed surfaces instead of the unit interval:

| domain | mesh | Laplacian |
|---|---|---|
| **Sphere** | icosphere, radius 1 (`MeshDomain.sphere`) | cotangent Laplace-Beltrami, `L = -M^{-1} S` |
| **Torus** | major radius 1, minor radius 0.3 (`MeshDomain.torus`) | same |

Both meshes have ~640 vertices. The PDEs (`DiffusionEquation`,
`ReactionDiffusion`, `BeamEquation`) are unchanged - they only see
`domain.laplacian`, so moving them onto a surface is just a different Domain.

**Task:** initial field `u(0)` on the mesh vertices -> evolved field `u(T)`.

**Models.** Same four as before:

| model | how it sees the surface |
|---|---|
| **Ours** | Laplace-Beltrami eigenbasis of the actual mesh - the geometry is built in |
| **CNO / FNO / LNO** | 1D models: the vertices are fed as a 1D sequence of length V |

The 1D baselines have no notion of the surface. To give them a fair chance the
vertices are put in a *raster* order (latitude bands on the sphere, rings
around the major circle on the torus, snaking so consecutive vertices are
neighbours), and every model gets the vertex xyz coordinates as extra input
channels. Their periodic FFT / convolution along this sequence is still not the
geometry of the surface: two vertices that are neighbours on the surface can
be far apart in the sequence. That gap is exactly what this benchmark
measures. A geometry-aware baseline (e.g. SFNO on a lat-lon grid of the
sphere, or a 2D FNO on the torus's periodic grid) would be a stronger
comparison; see the notes at the end.

In [ ]:
import os
import json
import time
import numpy as np
import scipy.sparse as sp
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import trimesh
from scipy.sparse.linalg import eigsh

from equations import (
    MeshDomain,
    DiffusionEquation,
    ReactionDiffusion,
    BeamEquation,
)
from model import HeatKernel, IntegralOperator, IntegralOperatorModel
from ref_models.cno import CNOOperator
from ref_models.fno import FNOOperator
from ref_models.lno import LNOOperator

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## Configuration

In [ ]:
SEED = 0
N_TOTAL = 128           # samples per (domain, system)
N_TRAIN = 96
N_EPOCHS = 1000
BATCH_SIZE = 32
LEARNING_RATE = 1e-3

# Meshes (~640 vertices each, so the two domains are directly comparable)
SPHERE_SUBDIVISIONS = 3             # icosphere: 642 vertices
TORUS_MAJOR_RADIUS = 1.0
TORUS_MINOR_RADIUS = 0.3
TORUS_MAJOR_SECTIONS = 40           # 40 x 16 = 640 vertices
TORUS_MINOR_SECTIONS = 16

# Vertex order seen by the 1D baselines: "raster" (neighbour-preserving
# snake through latitude bands / tube rings) or "native" (trimesh order,
# which for the icosphere is the subdivision hierarchy - spatially random).
VERTEX_ORDER = "raster"

# Initial conditions: random combinations of the lowest Laplace-Beltrami
# modes (constant mode excluded), the surface analogue of the low sine modes
# used on the interval. On the unit sphere 15 modes = degrees l = 1..3.
N_IC_MODES = 15

# Our model: Laplace-Beltrami eigenbasis settings (as in benchmark.ipynb)
N_MODES = 96
N_DIFFUSION_TIMES = 12              # heat-kernel time scales per layer
HIDDEN_CHANNELS = 32                # width of the message MLP
LATENT_CHANNELS = 8                 # channels between operator layers
N_OPERATOR_LAYERS = 4               # stacked integral-operator layers

# Append the vertex xyz coordinates as extra input channels for every model
# (the surface analogue of the 1D coordinate channel).
ADD_COORDINATE_CHANNELS = True

# Our IntegralOperator materialises a [batch, V, V, channels] pair tensor,
# which at V ~ 640 is ~25x larger than at V = 128. Gradients are accumulated
# over micro-batches of this size; the optimisation step is still over the
# full BATCH_SIZE, so the protocol is identical to the 1D benchmark.
OURS_MICRO_BATCH = 8

# Physical parameters per domain. The interval benchmark's values were
# tuned for Dirichlet sine modes with lambda ~ (k pi)^2 ~ 10..250; the low
# surface modes are much smaller (unit sphere: lambda = l(l+1) = 2..12, the
# torus's lowest 15 are ~1..12), so the coefficients are rescaled to give
# non-trivial dynamics. Both surfaces have similar low spectra, so they share
# the same values. The data cell prints ||u(T) - u(0)|| / ||u(0)|| so you
# can check each task really moves.
PDE_SETTINGS = {
    "Sphere": dict(D=0.1, k=0.5, T=1.0, EI=0.04, rho_A=1.0, T_beam=1.0),
    "Torus":  dict(D=0.1, k=0.5, T=1.0, EI=0.04, rho_A=1.0, T_beam=1.0),
}

# Quick end-to-end check (tiny data, 2 epochs): BENCHMARK_SMOKE=1
if os.environ.get("BENCHMARK_SMOKE") == "1":
    N_TOTAL, N_TRAIN, N_EPOCHS, BATCH_SIZE = 12, 8, 2, 4
    print("SMOKE TEST MODE")

## Domains

`MeshDomain` gives the cotangent Laplace-Beltrami operator of each mesh. The
vertices are permuted (faces remapped) before the domain is built, so every
model, every dataset and every plot share the same vertex order.

In [ ]:
def reorder_mesh(vertices, faces, order):
    # Permute vertices and remap faces; the mesh itself is unchanged.
    inverse = np.empty_like(order)
    inverse[order] = np.arange(len(order))
    return vertices[order], inverse[faces]


def snake_order(band, angle):
    # Sort by band, then by angle within the band, alternating direction
    # between bands so the sequence steps to a neighbour at every band change.
    key = np.where(band % 2 == 0, angle, -angle)
    return np.lexsort((key, band))


def sphere_raster(vertices):
    # Latitude bands of ~sqrt(V) rings, azimuth within each band.
    polar = np.arccos(np.clip(vertices[:, 2] / np.linalg.norm(vertices, axis=1), -1, 1))
    azimuth = np.arctan2(vertices[:, 1], vertices[:, 0])
    n_bands = int(round(np.sqrt(len(vertices))))
    band = np.minimum((polar / np.pi * n_bands).astype(int), n_bands - 1)
    return snake_order(band, azimuth)


def torus_raster(vertices, n_major):
    # One band per tube ring around the major circle, minor angle within it.
    major = np.mod(np.arctan2(vertices[:, 1], vertices[:, 0]), 2 * np.pi)
    rho = np.hypot(vertices[:, 0], vertices[:, 1])
    minor = np.arctan2(vertices[:, 2], rho - TORUS_MAJOR_RADIUS)
    band = np.round(major / (2 * np.pi) * n_major).astype(int) % n_major
    return snake_order(band, minor)


def build_domain(name):
    if name == "Sphere":
        mesh = trimesh.creation.icosphere(subdivisions=SPHERE_SUBDIVISIONS, radius=1.0)
        order_fn = sphere_raster
    else:
        mesh = trimesh.creation.torus(
            major_radius=TORUS_MAJOR_RADIUS,
            minor_radius=TORUS_MINOR_RADIUS,
            major_sections=TORUS_MAJOR_SECTIONS,
            minor_sections=TORUS_MINOR_SECTIONS,
        )
        order_fn = lambda v: torus_raster(v, TORUS_MAJOR_SECTIONS)

    vertices = np.asarray(mesh.vertices, dtype=float)
    faces = np.asarray(mesh.faces)

    if VERTEX_ORDER == "raster":
        vertices, faces = reorder_mesh(vertices, faces, order_fn(vertices))

    return MeshDomain(vertices, faces)


def sequence_locality(domain):
    # Mean surface distance between consecutive vertices in the sequence,
    # relative to the mean edge length: ~1 means the 1D baselines see true
    # neighbours side by side, >>1 means the sequence jumps around.
    p = domain.points
    edges = trimesh.Trimesh(p, domain.faces, process=False).edges_unique
    mean_edge = np.linalg.norm(p[edges[:, 0]] - p[edges[:, 1]], axis=1).mean()
    step = np.linalg.norm(np.diff(p, axis=0), axis=1).mean()
    return step / mean_edge


DOMAIN_NAMES = ["Sphere", "Torus"]
domains = {name: build_domain(name) for name in DOMAIN_NAMES}

for name, dom in domains.items():
    print(f"{name:7s} vertices {dom.n_points:4d}  faces {len(dom.faces):5d}  "
          f"area {dom.mass_diagonal.sum():.3f}  "
          f"sequence step / edge length {sequence_locality(dom):.2f} ({VERTEX_ORDER})")

## Laplace-Beltrami eigenpairs (for our model)

Generalised problem `S phi = lambda M phi`, solved through the symmetric
matrix `M^{-1/2} S M^{-1/2}` exactly as in `main.ipynb`. The eigenvectors are
then M-orthonormal (`Phi^T M Phi = I`), which `HeatKernel` relies on.
Both surfaces are closed, so `lambda_0 = 0` (the constant mode): unlike the
Dirichlet interval nothing is pinned to zero at a boundary.

In [ ]:
def make_mesh_eigenbasis(domain, n_modes=N_MODES):
    mass = domain.mass_diagonal.astype(np.float64)
    M_inv_sqrt = sp.diags(1.0 / np.sqrt(mass))
    L_symmetric = (M_inv_sqrt @ domain.stiffness @ M_inv_sqrt).tocsc()

    values, vectors = eigsh(L_symmetric, k=n_modes, sigma=-1e-6, which="LM")
    order = np.argsort(values)
    values = np.maximum(values[order], 0.0)
    vectors = np.asarray(M_inv_sqrt @ vectors[:, order])

    return values, vectors, mass


def make_diffusion_times(eigenvalues, n=N_DIFFUSION_TIMES):
    # As in benchmark.ipynb: 1/lambda_max .. 3/lambda_1, log-spaced. On a
    # closed surface lambda_0 = 0, so lambda_1 is the first non-zero one.
    first_nonzero = eigenvalues[eigenvalues > 1e-8][0]
    return list(
        np.logspace(np.log10(1.0 / eigenvalues[-1]),
                    np.log10(3.0 / first_nonzero), n)
    )


bases = {}
for name, dom in domains.items():
    values, vectors, mass = make_mesh_eigenbasis(dom)
    bases[name] = (values, vectors, mass)

    orthogonality_error = np.linalg.norm(
        vectors.T @ (mass[:, None] * vectors) - np.eye(N_MODES)
    )
    # -L phi should equal lambda phi with the domain's own Laplacian
    residual = np.linalg.norm(
        (-dom.laplacian) @ vectors - vectors * values[None, :]
    ) / np.linalg.norm(vectors * values[None, :])

    print(f"{name}: first eigenvalues {np.round(values[:10], 3)}")
    print(f"        lambda_max (mode {N_MODES}) {values[-1]:.1f}  "
          f"M-orthogonality error {orthogonality_error:.1e}  "
          f"eigen-residual {residual:.1e}")

print("unit sphere, exact l(l+1):", [l * (l + 1) for l in range(4) for _ in range(2 * l + 1)][:10])

## Data generation

Initial fields are random combinations of the lowest `N_IC_MODES`
Laplace-Beltrami modes (excluding the constant), scaled to max |u| = 1 - the
same recipe as the low sine modes on the interval. Note this uses the same
eigenbasis our model is built on, just as the interval benchmark's sine ICs
were the interval's Laplacian eigenvectors.

In [ ]:
def random_surface_field(rng, eigenvectors):
    coefficients = rng.normal(size=N_IC_MODES)
    field = eigenvectors[:, 1:N_IC_MODES + 1] @ coefficients
    return field / np.abs(field).max()


def make_pde_dataset(domain_name, equation_factory, T, seed=SEED):
    # Initial field u(0) -> final field u(T) on the mesh vertices.
    rng = np.random.default_rng(seed)
    _, eigenvectors, _ = bases[domain_name]
    u0_list, uT_list = [], []

    for _ in range(N_TOTAL):
        y0 = random_surface_field(rng, eigenvectors)

        equation = equation_factory()
        equation.set_initial_condition(y0=y0)
        solution = equation.solve([0.0, T], t_eval=[0.0, T])

        u0_list.append(solution.y[:, 0])
        uT_list.append(solution.y[:, -1])

    u0 = torch.tensor(np.array(u0_list), dtype=torch.float32).unsqueeze(-1)
    uT = torch.tensor(np.array(uT_list), dtype=torch.float32).unsqueeze(-1)
    return u0, uT


def make_beam_dataset(domain_name, EI, rho_A, T, seed=SEED):
    # Beam carries (w, wdot); evolve the displacement field.
    rng = np.random.default_rng(seed)
    domain = domains[domain_name]
    _, eigenvectors, _ = bases[domain_name]
    n = domain.n_points
    u0_list, uT_list = [], []

    for _ in range(N_TOTAL):
        w0 = random_surface_field(rng, eigenvectors)

        equation = BeamEquation(domain, EI=EI, rho_A=rho_A)
        equation.set_initial_condition(w0=w0, wdot0=np.zeros_like(w0))
        solution = equation.solve([0.0, T], t_eval=[0.0, T])

        u0_list.append(solution.y[:n, 0])
        uT_list.append(solution.y[:n, -1])

    u0 = torch.tensor(np.array(u0_list), dtype=torch.float32).unsqueeze(-1)
    uT = torch.tensor(np.array(uT_list), dtype=torch.float32).unsqueeze(-1)
    return u0, uT

In [ ]:
# datasets[(domain, system)] = (X, Y)
datasets = {}

for domain_name in DOMAIN_NAMES:
    dom = domains[domain_name]
    p = PDE_SETTINGS[domain_name]

    datasets[(domain_name, "Diffusion")] = make_pde_dataset(
        domain_name, lambda: DiffusionEquation(dom, D=p["D"]), T=p["T"]
    )
    datasets[(domain_name, "Reaction-Diffusion")] = make_pde_dataset(
        domain_name, lambda: ReactionDiffusion(dom, D=p["D"], k=p["k"]), T=p["T"]
    )
    datasets[(domain_name, "Beam")] = make_beam_dataset(
        domain_name, EI=p["EI"], rho_A=p["rho_A"], T=p["T_beam"]
    )

for (domain_name, system), (X, Y) in datasets.items():
    change = (torch.norm(Y - X) / torch.norm(X)).item()
    print(f"{domain_name:7s} {system:20s} X {tuple(X.shape)}  Y {tuple(Y.shape)}  "
          f"||u(T)-u(0)||/||u(0)|| = {change:.3f}  finite: {torch.isfinite(Y).all().item()}")

### Normalization

Training-set statistics only, errors reported in normalized space - same as
`benchmark.ipynb`. The xyz coordinates (scaled to [-1, 1]) are appended after
normalization.

In [ ]:
def normalize(X, Y, n_train=N_TRAIN):
    x_mean, x_std = X[:n_train].mean(), X[:n_train].std()
    y_mean, y_std = Y[:n_train].mean(), Y[:n_train].std()
    x_std = torch.clamp(x_std, min=1e-8)
    y_std = torch.clamp(y_std, min=1e-8)
    return (X - x_mean) / x_std, (Y - y_mean) / y_std


def add_coordinate_channels(X, points):
    coordinates = torch.tensor(points / np.abs(points).max(), dtype=X.dtype)
    coordinates = coordinates.unsqueeze(0).expand(X.shape[0], -1, -1)
    return torch.cat([X, coordinates], dim=-1)


normalized = {}
for (domain_name, system), (X, Y) in datasets.items():
    X, Y = normalize(X, Y)
    if ADD_COORDINATE_CHANNELS:
        X = add_coordinate_channels(X, domains[domain_name].points)
    normalized[(domain_name, system)] = (X, Y)

datasets = normalized

print("normalized using training-set statistics"
      + (", xyz channels appended" if ADD_COORDINATE_CHANNELS else ""))
for (domain_name, system), (X, Y) in datasets.items():
    print(f"  {domain_name:7s} {system:20s} X {tuple(X.shape)}  Y {tuple(Y.shape)}")

## Model builders

Identical hyperparameters to `benchmark.ipynb`. Ours is built on the mesh's
eigenbasis; the baselines get `n_points = V` and treat the vertex sequence as
a 1D signal (LNO's coordinate is the normalised sequence index, since there
is no physical 1D axis).

In [ ]:
def build_ours(in_channels, out_channels, domain_name):
    values, vectors, weights = bases[domain_name]
    times = make_diffusion_times(values)

    layers = []

    for layer_index in range(N_OPERATOR_LAYERS):
        layer_in = in_channels if layer_index == 0 else LATENT_CHANNELS
        layer_out = (
            out_channels
            if layer_index == N_OPERATOR_LAYERS - 1
            else LATENT_CHANNELS
        )

        kernel = HeatKernel(
            eigenvalues=torch.tensor(values, dtype=torch.float32),
            eigenvectors=torch.tensor(vectors, dtype=torch.float32),
            mass=weights,
            in_channels=layer_in,
            out_channels=layer_out,
            diffusion_times=times,
        )
        layers.append(
            IntegralOperator(kernel, hidden_channels=HIDDEN_CHANNELS)
        )

    return IntegralOperatorModel(layers)


def build_cno(in_channels, out_channels, domain_name):
    return CNOOperator(
        n_points=domains[domain_name].n_points,
        in_channels=in_channels,
        out_channels=out_channels,
        N_layers=3,
        N_res=2,
        N_res_neck=2,
        channel_multiplier=16,
    )


def build_fno(in_channels, out_channels, domain_name):
    return FNOOperator(
        n_points=domains[domain_name].n_points,
        in_channels=in_channels,
        out_channels=out_channels,
        modes=16,
        width=64,
    )


def build_lno(in_channels, out_channels, domain_name):
    n_points = domains[domain_name].n_points
    return LNOOperator(
        n_points=n_points,
        in_channels=in_channels,
        out_channels=out_channels,
        modes=16,
        width=4,
        coords=np.linspace(0.0, 1.0, n_points),
    )


MODEL_BUILDERS = {
    "Ours": build_ours,
    "CNO": build_cno,
    "FNO": build_fno,
    "LNO": build_lno,
}

# Micro-batch per model (gradient accumulation; the optimiser step is always
# over BATCH_SIZE). CNO uses BatchNorm, so baselines keep the full batch.
MICRO_BATCH = {"Ours": OURS_MICRO_BATCH}

## Training loop

Same as `benchmark.ipynb` - Adam, mini-batches, 1000 epochs, learning rate
halved every 250 epochs - plus gradient accumulation over micro-batches so
our O(V^2) model fits in GPU memory. Accumulating `loss * len(chunk) /
len(batch)` gives exactly the full-batch MSE gradient.

In [ ]:
def relative_l2(prediction, target):
    return (
        torch.norm(prediction - target) / torch.norm(target)
    ).item()


@torch.no_grad()
def predict(model, X, micro_batch):
    return torch.cat([model(chunk) for chunk in X.split(micro_batch)])


def train_model(model, X, Y, micro_batch=BATCH_SIZE, n_epochs=N_EPOCHS, seed=SEED):
    torch.manual_seed(seed)
    model = model.to(device)

    X_train, Y_train = X[:N_TRAIN].to(device), Y[:N_TRAIN].to(device)
    X_test, Y_test = X[N_TRAIN:].to(device), Y[N_TRAIN:].to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=250, gamma=0.5
    )

    history = {"train_loss": [], "test_error": []}
    n_batches = max(1, N_TRAIN // BATCH_SIZE)

    for epoch in range(n_epochs):
        model.train()
        permutation = torch.randperm(N_TRAIN, device=device)
        epoch_loss = 0.0

        for b in range(n_batches):
            index = permutation[b * BATCH_SIZE:(b + 1) * BATCH_SIZE]
            optimizer.zero_grad()

            for chunk in index.split(micro_batch):
                loss = nn.functional.mse_loss(
                    model(X_train[chunk]), Y_train[chunk]
                ) * (len(chunk) / len(index))
                loss.backward()
                epoch_loss += loss.item()

            optimizer.step()

        scheduler.step()

        model.eval()
        test_error = relative_l2(predict(model, X_test, micro_batch), Y_test)

        history["train_loss"].append(epoch_loss / n_batches)
        history["test_error"].append(test_error)

    return history

## Run the benchmark

4 models x 2 domains x 3 systems x 1000 epochs. A model that runs out of GPU
memory is recorded as NaN instead of stopping the whole run.

In [ ]:
results = {}
histories = {}
trained_models = {}

for (domain_name, system), (X, Y) in datasets.items():
    key = (domain_name, system)
    in_channels = X.shape[-1]
    out_channels = Y.shape[-1]
    results[key] = {}
    histories[key] = {}
    trained_models[key] = {}

    print(f"\n=== {domain_name} / {system} "
          f"(V={X.shape[1]}, in={in_channels}, out={out_channels}) ===", flush=True)

    for model_name, builder in MODEL_BUILDERS.items():
        torch.manual_seed(SEED)
        model = builder(in_channels, out_channels, domain_name)
        n_parameters = sum(p.numel() for p in model.parameters())
        micro_batch = MICRO_BATCH.get(model_name, BATCH_SIZE)

        start = time.time()
        try:
            history = train_model(model, X, Y, micro_batch=micro_batch)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            print(f"  {model_name:5s} OUT OF MEMORY - lower OURS_MICRO_BATCH", flush=True)
            history = {"train_loss": [float("nan")], "test_error": [float("nan")]}
        elapsed = time.time() - start

        final_error = history["test_error"][-1]
        best_error = min(history["test_error"])

        results[key][model_name] = {
            "final_error": final_error,
            "best_error": best_error,
            "n_parameters": n_parameters,
            "seconds": elapsed,
        }
        histories[key][model_name] = history
        trained_models[key][model_name] = model

        peak = (f"  peak GPU {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB"
                if torch.cuda.is_available() else "")
        print(f"  {model_name:5s} params={n_parameters:8d}  "
              f"final rel L2 {final_error:.4f}  "
              f"best {best_error:.4f}  ({elapsed:.0f}s){peak}", flush=True)

## Results table

Test relative L2 error after training (lower is better); the best model for
each row is marked with `*`.

In [ ]:
model_names = list(MODEL_BUILDERS.keys())

header = f"{'Domain':<8}{'System':<20}" + "".join(f"{m:>12}" for m in model_names)
print(header)
print("-" * len(header))

for (domain_name, system) in datasets:
    row = f"{domain_name:<8}{system:<20}"
    errors = {m: results[(domain_name, system)][m]["final_error"] for m in model_names}
    finite = {m: e for m, e in errors.items() if np.isfinite(e)}
    best_model = min(finite, key=finite.get) if finite else None

    for model_name in model_names:
        mark = "*" if model_name == best_model else " "
        row += f"{errors[model_name]:>11.4f}{mark}"
    print(row)

print("\nParameter counts")
print(f"{'Domain':<8}{'System':<20}" + "".join(f"{m:>12}" for m in model_names))
for (domain_name, system) in datasets:
    row = f"{domain_name:<8}{system:<20}"
    for model_name in model_names:
        row += f"{results[(domain_name, system)][model_name]['n_parameters']:>12d}"
    print(row)

In [ ]:
try:
    import pandas as pd
    from IPython.display import display

    table = pd.DataFrame(
        {
            model_name: {
                key: results[key][model_name]["final_error"] for key in datasets
            }
            for model_name in model_names
        }
    )
    table.index = pd.MultiIndex.from_tuples(table.index, names=["Domain", "System"])
    display(table.style.format("{:.4f}").highlight_min(axis=1, color="#c6efce"))
except ImportError:
    print("pandas not installed - see the printed table above instead.")

In [ ]:
# Save the numbers next to the executed notebook so runs can be compared
# without re-opening notebooks.
os.makedirs("results", exist_ok=True)
run_id = os.environ.get("SLURM_JOB_ID", "local")
results_path = f"results/manifold_results_{run_id}.json"

with open(results_path, "w") as f:
    json.dump(
        {f"{d}/{s}": v for (d, s), v in results.items()},
        f,
        indent=2,
    )
print("saved", results_path)

## Loss curves

One figure per domain: training loss (top) and test error (bottom) per system.

In [ ]:
colors = {"Ours": "#d62728", "CNO": "#1f77b4", "FNO": "#2ca02c",
          "LNO": "#ff7f0e"}

systems = ["Diffusion", "Reaction-Diffusion", "Beam"]

for domain_name in DOMAIN_NAMES:
    fig, axes = plt.subplots(2, len(systems), figsize=(4.2 * len(systems), 7.5))

    for column, system in enumerate(systems):
        ax_loss, ax_error = axes[0, column], axes[1, column]

        for model_name in model_names:
            history = histories[(domain_name, system)][model_name]
            ax_loss.semilogy(history["train_loss"], label=model_name,
                             color=colors[model_name], linewidth=1.2)
            ax_error.semilogy(history["test_error"], label=model_name,
                              color=colors[model_name], linewidth=1.2)

        ax_loss.set_title(f"{domain_name} / {system}")
        ax_loss.set_xlabel("Epoch")
        ax_error.set_xlabel("Epoch")
        if column == 0:
            ax_loss.set_ylabel("Training MSE")
            ax_error.set_ylabel("Test relative L2")
        ax_loss.legend(fontsize=8)
        ax_loss.grid(alpha=0.3)
        ax_error.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

## Example predictions on the surface

One held-out test sample per (domain, system): ground truth and each model's
prediction, coloured on the mesh vertices with a shared colour scale per row.

In [ ]:
sample = 0
columns = ["Ground truth"] + model_names

for domain_name in DOMAIN_NAMES:
    points = domains[domain_name].points
    fig = plt.figure(figsize=(3.2 * len(columns), 3.0 * len(systems)))

    for row, system in enumerate(systems):
        X, Y = datasets[(domain_name, system)]
        fields = {"Ground truth": Y[N_TRAIN + sample, :, 0].numpy()}

        for model_name in model_names:
            model = trained_models[(domain_name, system)][model_name]
            model.eval()
            with torch.no_grad():
                fields[model_name] = model(
                    X[N_TRAIN + sample:N_TRAIN + sample + 1].to(device)
                )[0, :, 0].cpu().numpy()

        vmin, vmax = fields["Ground truth"].min(), fields["Ground truth"].max()

        for column, name in enumerate(columns):
            ax = fig.add_subplot(len(systems), len(columns),
                                 row * len(columns) + column + 1, projection="3d")
            ax.scatter(points[:, 0], points[:, 1], points[:, 2],
                       c=fields[name], cmap="RdBu_r", vmin=vmin, vmax=vmax, s=8)
            ax.set_box_aspect(np.ptp(points, axis=0))
            ax.set_axis_off()
            error = "" if name == "Ground truth" else (
                f"\nrel L2 {np.linalg.norm(fields[name] - fields['Ground truth']) / np.linalg.norm(fields['Ground truth']):.3f}"
            )
            ax.set_title(f"{system}: {name}{error}" if column == 0 else f"{name}{error}",
                         fontsize=9)

    fig.suptitle(domain_name)
    plt.tight_layout()
    plt.show()

## Notes for interpreting the results

- **Where our model should win.** The surfaces are closed and curved; our
  basis is the mesh's own Laplace-Beltrami eigenbasis, while the baselines see
  an arbitrary 1D ordering. Diffusion is exactly a heat-kernel filter in this
  basis, so it is the cleanest test of the geometric prior.
- **Vertex order matters for the baselines.** Rerun with
  `VERTEX_ORDER = "native"` to see how much of the baselines' accuracy depends
  on the hand-made raster order; ours is unaffected by any permutation.
- **ICs live in our basis.** Inputs are combinations of the lowest
  `N_IC_MODES` eigenvectors, which our model represents exactly (as the sine
  ICs were on the interval). A tougher test would use ICs with energy
  above mode `N_MODES`, e.g. random vertex noise smoothed by a short heat flow
  as in `main.ipynb`.
- **Stronger baselines.** For a geometry-aware comparison, add SFNO
  (spherical harmonics on a lat-lon grid) for the sphere and a 2D FNO on the
  torus's periodic (major x minor) grid.